# Análise de Crescimento Populacional (IBGE 2010 vs 2022)

Este notebook executa o tratamento, agregação por Estado (UF) e Município, e visualização dos dados do Censo Demográfico.

In [1]:

import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

df = pd.read_excel('populacao_2010_2022.xlsx')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'populacao_2010_2022.xlsx'

In [ ]:

col_2010 = 'POP_2010'
col_2022 = 'POP_2022'

pop_estado = (
    df.groupby(["COD. UF", "UF"], as_index=False)
      .agg(
          populacao_2010=(col_2010, "sum"),
          populacao_2022=(col_2022, "sum"),
      )
)

pop_estado["crescimento_absoluto"] = pop_estado["populacao_2022"] - pop_estado["populacao_2010"]

pop_estado = pop_estado.sort_values(by="crescimento_absoluto", ascending=False)

pop_estado.to_csv("populacao_por_estado.csv", sep=";", index=False, encoding="utf-8-sig")
pop_estado

In [ ]:

col_muni_nome = 'NOME DO MUNICÍPIO' if 'NOME DO MUNICÍPIO' in df.columns else 'MUNICÍPIO'

pop_municipio = df.copy()
pop_municipio["crescimento_absoluto"] = pop_municipio[col_2022] - pop_municipio[col_2010]
pop_municipio = pop_municipio.sort_values(by="crescimento_absoluto", ascending=False)

pop_municipio.to_csv("populacao_por_municipio.csv", sep=";", index=False, encoding="utf-8-sig")
pop_municipio.head(10)

In [ ]:

pop_estado_plot = pop_estado.rename(columns={
    "populacao_2010": "2010",
    "populacao_2022": "2022"
})

top = pop_estado_plot.head(10).melt(
    id_vars="UF",
    value_vars=["2010", "2022"],
    var_name="ano",
    value_name="populacao",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top, x="UF", y="populacao", hue="ano", ax=ax)
ax.set_title("10 UFs com maior crescimento absoluto: 2010 vs 2022")
ax.set_ylabel("Habitantes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()